In [ ]:
"""
Convert the subsetted NextGen v2.2 hydrofabric geopackage to the partitioned model json file used in
the KF codebase.

Written by Quinn Russell
"""

import json
import geopandas as gpd

In [23]:
# read original json format to get partitions
with open("../1/travis-county-lo-res-model-partitioned.json", "r", encoding="utf-8") as f:
    nwm_models = json.load(f)

In [ ]:
# read gpkg to get topology, length, and geometry
flowpaths = gpd.read_file("../3/tx_subset.gpkg", layer="flowpaths")
nexus = gpd.read_file("../3/tx_subset.gpkg", layer="nexus")

In [28]:
# Map NWM IDs to NextGen IDs
with open("../1/hf2.2_ref_hf_map.json", "r", encoding="utf-8") as f:
    nwm_to_ngen_mapping = json.load(f)

In [ ]:
def convert_nwm_to_nextgen(reaches: list, mapping: dict) -> list:
    """Convert NWM IDs to NextGen WB IDs.

    Args:
        reaches (list): NWM reaches to convert
        mapping (dict): Mapping that looks like
            {"wb_id_1": ["nwm_id1", "nwm_id2"], "wb_id_2": ["nwm_id3", "nwm_id4", "nwm_id5], ...}

    Returns:
        list: NextGen v2.2 wb-ids
    """
    reach_floats = [float(reach) for reach in reaches]
    reverse_map = set()
    for ngen_id, nwm_ids in mapping.items():
        for nwm_id in nwm_ids:
            if nwm_id in reach_floats:
                reverse_map.add(ngen_id)

    ngen_reaches = list(reverse_map)

    wb_ids = [i.replace("cat-", "wb-", 1) if i.startswith("cat-") else i for i in ngen_reaches]

    return wb_ids

In [67]:
# This is not a particularly efficient cell but it should run in about 8 seconds

individual_models = {}
for i in range(4): # 4 models
    # sorted to improve lookup speed
    nwm_reaches = nwm_models["models"][str(i)]["model"]["reach_ids"]
    ngen_reach_ids = sorted(convert_nwm_to_nextgen(nwm_reaches, nwm_to_ngen_mapping))

    num_reaches = len(ngen_reach_ids)
    startnodes = list(range(num_reaches))
    endnodes = []

    # get endnodes
    for reach in ngen_reach_ids:
        to_nexus = flowpaths[flowpaths["id"]==reach]["toid"].values[0]
        next_reach = nexus[nexus["id"]==to_nexus]["toid"].values[0]
        if next_reach not in ngen_reach_ids:
            next_reach = reach # this is how KF handles terminal reaches
        endreach_id = ngen_reach_ids.index(next_reach)
        endnodes.append(endreach_id)

    # get lengths
    dx = []
    for reach in ngen_reach_ids:
        length = flowpaths[flowpaths["id"]==reach]["lengthkm"].values[0] * 1000 # km to m
        dx.append(length)

    # get paths
    paths = []
    for reach in ngen_reach_ids:
        linestring = flowpaths[flowpaths["id"]==reach]["geometry"].values[0]
        path = [list(pt) for pt in linestring.coords]
        paths.append(path)

    individual_models[str(i)] = {
        "name": str(i),
        "datetime": "2024-12-31T23:58:00+00:00",
        "timedelta": "P0DT0H2M0S",
        "reach_ids": ngen_reach_ids,
        "startnodes": startnodes,
        "endnodes": endnodes,
        "K": [0] * num_reaches, # dummy K and x values, will be passed in from t-route
        "X": [0] * num_reaches,
        "o_t": [0.0] * num_reaches,
        "dx": dx,
        "paths": paths
    }

In [68]:
all_models = {}
for i in range(4):
    all_models[str(i)] = {
        "model": individual_models[str(i)],
        "sinks": [],
        "sources": []
    }

full_partitioned_model = {
    "models": all_models,
    "connections": {}
}

In [69]:
with open("tx_ngen_model_partitioned.json", "w", encoding="utf-8") as model_file:
    json.dump(full_partitioned_model, model_file, indent=4)